In [ ]:
#Import needed   libraries
import pandas as pd
import numpy as np
import string
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv('tech_skills.csv')
# Check columns and data types
print(df.info())

# Summary statistics
print(df.describe())


In [3]:
df.head()

,Skills,Tools,Interest,Education Level,Years of Experience,Recommended Career
0,"Power BI, Python, Azure, React, Tableau, Deep ...","Google Cloud, Docker",IoT,Postgraduate,7,Embedded Systems Engineer
1,"SQL, Machine Learning, React, PyTorch, Tableau...","Postman, VS Code, Azure Portal, GitLab",Cloud,Postgraduate,6,DevOps Engineer
2,"Java, Figma, HTML, Azure, Deep Learning, REST ...","GitLab, GitHub, PyCharm",Web Development,Graduate,3,Backend Developer
3,"Embedded Systems, Spark, Linux, Statistics, De...","Kubernetes, GitHub, VS Code",Cybersecurity,Postgraduate,5,Security Engineer
4,"Node.js, Power BI, React, Figma, Git, UI/UX","Jira, Figma, Kubernetes",AI/ML,Graduate,7,AI Engineer


In [4]:
df.drop(columns=['Years of Experience'], inplace=True)
df.rename(columns={'Education Level': 'Education'}, inplace=True)
df.rename(columns={'Interest': 'Interests'}, inplace=True)
print(df['Skills'].unique())


['Power BI, Python, Azure, React, Tableau, Deep Learning'
 'SQL, Machine Learning, React, PyTorch, Tableau, Kubernetes, Java'
 'Java, Figma, HTML, Azure, Deep Learning, REST APIs' ...
 'React, JavaScript, SQL, Data Analysis'
 'Statistics, Deep Learning, Tableau, Java, Machine Learning'
 'AWS, Spark, Node.js, React, NLP, Cybersecurity']


In [5]:
#df.drop(columns=['CandidateID'], inplace=True)

In [6]:
#Data Pre processing
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
df['Skills'] = df['Skills'].str.lower()
df['Interests'] = df['Interests'].str.lower()
df['Recommended_Career'] = df['Recommended Career'].str.lower()
df['Tools'] = df['Tools'].str.lower()
df['Education'] = df['Education'].str.lower()

# Optional: just for readability
df['Skills'] = df['Skills'].str.replace(';', ' > ')
df['Interests'] = df['Interests'].str.replace(';', ' > ')
df['Recommended_Career'] = df['Recommended_Career'].str.replace(';', ' > ')
df['Education'] = df['Education'].str.replace(f'[{string.punctuation}]', '',  regex=True)
df['Tools'] = df['Tools'].str.replace(';', ' > ')
print(df.duplicated().sum())

0


In [9]:
df.head(15)
cleaned_df = df
cleaned_df.to_csv('tech_skills_cleaned.csv')

In [7]:
def Smart_Career_Recommendation(career_df):
    '''
    A smart creer recommendation systtem that recommends to users a career path they should based on some personalised qualities and interest of the user
    :param df: A dataset that coontains some personalised features or interests of users. It has columns like 'Knowledge', 'Interest', etc
    :return:
    list:  list of recommended career  paths
    '''
    tfidf_skills = TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_df=0.9,min_df=2,sublinear_tf=True)
    tfidf_interests = TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_df=0.9,min_df=2,sublinear_tf=True)
    tfidf_edu = TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_df=0.9,min_df=2,sublinear_tf=True)
    tfidf_tools = TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_df=0.9,min_df=2,sublinear_tf=True)
#Matrices for the features of the dataset
    skills_matrix = tfidf_skills.fit_transform(df['Skills'])
    interests_matrix = tfidf_interests.fit_transform(df['Interests'])
    edu_matrix = tfidf_edu.fit_transform(df['Education'])
    tools_matrix = tfidf_tools.fit_transform(df['Tools'])
    print(f'model trained on {len(career_df)} data points')
    while True:
        print("\nEnter 'quit' anytime to exit.\n")
        #Allowed   use  inputs for prediction
        u_interests = input('Enter Area of  Interests in the tech field: ').strip().lower()
        if u_interests.lower() == 'quit':
            print("Exiting...")
            break
        u_skills = input('Enter Skills  you have in the tech field: ').strip().lower()
        if u_skills.lower() == 'quit':
            print("Exiting...")
            break
        u_education = input('Enter Education  level in study: ')
        if u_education.lower() == 'quit':
            print("Exiting...")
            break
        u_tools = input('Enter any tech tool you are conversant with: ')
        if u_tools.lower() == 'quit':
            print("Exiting...")
            break

            #Vectorise use inputs using the tfid of the dataset
        u_skills_vec = tfidf_skills.transform([u_skills])
        u_interest_vec = tfidf_interests.transform([u_interests])
        u_edu_vec = tfidf_edu.transform([u_education])
        u_tools_vec = tfidf_tools.transform([u_tools])

        # Compute similarity separately
        skills_sim = cosine_similarity(u_skills_vec, skills_matrix).flatten()
        interest_sim = cosine_similarity(u_interest_vec, interests_matrix).flatten()
        edu_sim = cosine_similarity(u_edu_vec, edu_matrix).flatten()
        tools_sim = cosine_similarity(u_tools_vec, tools_matrix).flatten()
        final_score = (0.4 * skills_sim) + (0.3 * interest_sim) + (0.1 * edu_sim) + (0.2 * tools_sim)
        sim_score = list(enumerate(final_score))
        sim_score= sorted(sim_score, key=lambda x: x[1], reverse=True)[1:11]
        top_idx = [i[0] for i in sim_score]
        rec_career = df.iloc[top_idx]['Recommended_Career'].values
        recommended_df = pd.DataFrame({
            'Career': rec_career,
            'Skills Match %': (skills_sim[top_idx] * 100).round(2),
            'Interest Match %': (interest_sim[top_idx] * 100).round(2),
            'Education Match %': (edu_sim[top_idx] * 100).round(2),
            'Tools Match %': (tools_sim[top_idx] * 100).round(2),
            'Final Score %': (final_score[top_idx] * 100).round(2)
            })
        recommended_df = recommended_df.sort_values('Final Score %', ascending=False).drop_duplicates(subset=['Career']).reset_index(drop=True)
        print("\n Top Recommended Career Paths for You:")
        print(recommended_df.to_string(index=False))
        plt.figure(figsize=(8, 4))
        sns.barplot(x='Career', y='Final Score %', data=recommended_df)
        plt.title("Top Career Recommendations")
        plt.xticks(rotation=75)
        plt.tight_layout()
        plt.show()
    sim_df = pd.DataFrame({
    "Skills": skills_sim,
    "Interests": interest_sim,
    "Education": edu_sim,
    'Tools': tools_sim
    })
    sns.heatmap(sim_df.corr(), annot=True)
    plt.title("Similarity Factors Correlation")
    plt.show()
    print('Goodluck in your recommended career paths')




In [8]:
Smart_Career_Recommendation(df)

model trained on 3000 data points

Enter 'quit' anytime to exit.

Exiting...


UnboundLocalError: cannot access local variable 'skills_sim' where it is not associated with a value

In [10]:
import os
import joblib
#Here saved the vectorizers and the matrices for the trained model or the dataset trained with it in order to compute with similarity in the web app after collecting user inputs
tfidf_skills = TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_df=0.9,min_df=2,sublinear_tf=True)
tfidf_interests = TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_df=0.9,min_df=2,sublinear_tf=True)
tfidf_edu = TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_df=0.9,min_df=2,sublinear_tf=True)
tfidf_tools = TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_df=0.9,min_df=2,sublinear_tf=True)

skills_matrix = tfidf_skills.fit_transform(df['Skills'])
interests_matrix = tfidf_interests.fit_transform(df['Interests'])
edu_matrix = tfidf_edu.fit_transform(df['Education'])
tools_matrix = tfidf_tools.fit_transform(df['Tools'])

os.makedirs('smart_career_models', exist_ok=True)
joblib.dump(tfidf_skills, "smart_career_models/tfidf_skills.pkl")
joblib.dump(tfidf_interests, "smart_career_models/tfidf_interests.pkl")
joblib.dump(tfidf_edu, "smart_career_models/tfidf_edu.pkl")
joblib.dump(tfidf_tools, "smart_career_models/tfidf_tools.pkl")


joblib.dump(skills_matrix, "smart_career_models/skills_matrix.pkl")
joblib.dump(interests_matrix, "smart_career_models/interests_matrix.pkl")
joblib.dump(edu_matrix, "smart_career_models/edu_matrix.pkl")
joblib.dump(tools_matrix, "smart_career_models/tools_matrix.pkl")

['smart_career_models/tools_matrix.pkl']